# V6.5 — Camber Compose + QA Launcher

Đây là **bước tiếp theo sau `KAGGLE_V6P5_CAMBER_ZERO_DISK_OFFLOAD.ipynb`**.

Notebook này không kéo dataset lớn về Kaggle. Kaggle chỉ upload một worker nhỏ lên Camber và submit job.

Worker Camber sẽ:

1. kiểm tra source staging đã có đủ;
2. lấy `V6.2E TRAIN background pool` nhỏ đã tạo ở bước trước;
3. đọc Open Images từng class một:
   - Balloon
   - Kite
   - Plastic bag
   - Bird
   - Person
4. chỉ dùng object có segmentation mask để tạo synthetic sạch;
5. tạo:
   - **positive relation samples**: object gần electrical structure → `foreign_object`;
   - **hard-negative relation samples**: object ở xa structure → không gắn `foreign_object`;
6. đọc manifest của MPCD và Power Equipment trực tiếp từ tar, không extract full;
7. exact-SHA audit các direct external sources với toàn bộ V6.2E manifest;
8. tạo review manifest + contact sheets;
9. đóng gói synthetic overlay;
10. upload toàn bộ QA nhỏ + overlay về Camber Stash;
11. cleanup local temp trên Camber node.

## Không promotion tự động

Kết quả bước này vẫn là **candidate staging**.

Sau khi xem contact sheets / review manifest mới freeze V6.5 canonical candidate và sanity-train.

In [ ]:
# ==============================================================================
# 0. AUTH + CONFIG
# ==============================================================================
from pathlib import Path
import os, sys, json, shutil, subprocess, textwrap

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
JOB_ROOT = f"{STAGE_ROOT}/jobs"
COMPOSE_ROOT = f"{STAGE_ROOT}/compose_qa"

NODE_SIZE = "MEDIUM"

# Synthetic policy
SYNTH_POS_PER_SUBTYPE = 90
SYNTH_NEG_PER_SUBTYPE = 30
SEED = 20261007

key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("Bật CAMBER_API_KEY trong Kaggle Secrets.")

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"
if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH", "")
)
CAMBER = shutil.which("camber") or str(camber_bin)

assert Path(CAMBER).exists(), "Camber CLI not found"

RAM_TMP = Path("/dev/shm/v65_compose_launcher")
RAM_TMP.mkdir(parents=True, exist_ok=True)

print("CAMBER:", CAMBER)
print("STAGE_ROOT:", STAGE_ROOT)
print("COMPOSE_ROOT:", COMPOSE_ROOT)

In [ ]:
# ==============================================================================
# 1. COMPOSE + QA WORKER
# ==============================================================================
COMPOSE_WORKER = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, tarfile, zipfile, io, random, math
import pandas as pd
import numpy as np

ROOT = Path("/tmp/v65_compose_worker")
SRC = ROOT / "src"
OUT = ROOT / "out"
SYN = OUT / "synthetic_foreign"
QA = OUT / "qa"

for p in [SRC, OUT, SYN, QA]:
    p.mkdir(parents=True, exist_ok=True)

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
COMPOSE_ROOT = f"{STAGE_ROOT}/compose_qa"

V6ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
V6TAG = "v6p2e_23562a45b343"

OI_SUBTYPES = ["balloon", "kite", "plastic_bag", "bird", "person"]
SEED = 20261007
POS_PER = 90
NEG_PER = 30

random.seed(SEED)
np.random.seed(SEED)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "pillow", "pyyaml",
    "fiftyone==1.22.1", "tqdm"
])

from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
import yaml
import fiftyone.utils.openimages as fouo

def ensure_camber():
    c = shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    if not Path(c).exists():
        subprocess.check_call(
            "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
            shell=True,
        )
        os.environ["PATH"] = f"{Path.home()}/.camber/bin:" + os.environ.get("PATH","")
        c = shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    assert Path(c).exists()
    return c

CAMBER = ensure_camber()

def run(cmd):
    r = subprocess.run(cmd, text=True, capture_output=True, env=os.environ.copy())
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[:10000])
    return r

def get(remote, local):
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    run([CAMBER, "stash", "cp", remote, str(local)])
    return local

def put(local, remote):
    run([CAMBER, "stash", "cp", str(local), remote])

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(1024*1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def read_yolo_text(txt):
    rows = []
    for line in str(txt).splitlines():
        p = line.strip().split()
        if len(p) < 5:
            continue
        try:
            rows.append((int(float(p[0])), *map(float, p[1:5])))
        except Exception:
            pass
    return rows

def read_yolo_file(p):
    if not Path(p).exists():
        return []
    return read_yolo_text(Path(p).read_text(errors="ignore"))

def write_yolo(p, rows):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    txt = "\n".join(
        f"{int(c)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}"
        for c,x,y,w,h in rows if w > 0 and h > 0
    )
    Path(p).write_text(txt + ("\n" if txt else ""))

def xyxy_to_yolo(box):
    x1,y1,x2,y2 = map(float, box)
    x1 = max(0,min(1,x1)); y1 = max(0,min(1,y1))
    x2 = max(0,min(1,x2)); y2 = max(0,min(1,y2))
    return [(x1+x2)/2, (y1+y2)/2, max(0,x2-x1), max(0,y2-y1)]

def box_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1])
    x2=min(a[2],b[2]); y2=min(a[3],b[3])
    iw=max(0,x2-x1); ih=max(0,y2-y1)
    inter=iw*ih
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    u=aa+bb-inter
    return inter/u if u>0 else 0.0

def anchor_boxes(rows):
    out=[]
    for c,xc,yc,w,h in rows:
        if c != 0:
            continue
        out.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
    return out

def resize_rgba(obj, target_long):
    ow,oh = obj.size
    if max(ow,oh) <= 0:
        return obj
    s = target_long / max(ow,oh)
    nw=max(2,int(round(ow*s)))
    nh=max(2,int(round(oh*s)))
    return obj.resize((nw,nh), Image.Resampling.LANCZOS)

def object_target_scale(subtype, anchor_long, rng, image_min):
    if subtype == "person":
        rel = rng.uniform(0.45, 0.90)
    elif subtype in {"balloon","kite"}:
        rel = rng.uniform(0.20, 0.55)
    elif subtype == "bird":
        rel = rng.uniform(0.10, 0.28)
    else:
        rel = rng.uniform(0.12, 0.35)

    return max(12, min(int(anchor_long*rel), int(image_min*0.30)))

def place_positive(bg, obj, anchor, subtype, rng):
    W,H = bg.size
    x1,y1,x2,y2 = anchor
    acx=(x1+x2)/2*W
    acy=(y1+y2)/2*H
    aw=(x2-x1)*W
    ah=(y2-y1)*H

    obj = resize_rgba(
        obj,
        object_target_scale(subtype, max(aw,ah), rng, min(W,H))
    )

    if obj.width >= W or obj.height >= H:
        return None

    if subtype == "person":
        dx=rng.uniform(-0.9,0.9)*max(aw,40)
        dy=rng.uniform(0.30,1.20)*max(ah,40)
    else:
        dx=rng.uniform(-0.75,0.75)*max(aw,35)
        dy=rng.uniform(-0.65,0.65)*max(ah,35)

    cx=acx+dx; cy=acy+dy
    px=max(0,min(W-obj.width,int(round(cx-obj.width/2))))
    py=max(0,min(H-obj.height,int(round(cy-obj.height/2))))

    box=[px/W, py/H, (px+obj.width)/W, (py+obj.height)/H]

    # Keep a meaningful relation to at least one electrical anchor.
    if box_iou(box, anchor) == 0:
        # distance criterion if no overlap
        bx=(box[0]+box[2])/2; by=(box[1]+box[3])/2
        ax=(anchor[0]+anchor[2])/2; ay=(anchor[1]+anchor[3])/2
        d=math.sqrt((bx-ax)**2+(by-ay)**2)
        if d > 0.25:
            return None

    out=bg.convert("RGBA")
    out.alpha_composite(obj,(px,py))
    return out.convert("RGB"), box

def place_negative(bg, obj, anchors, subtype, rng):
    W,H = bg.size
    # scale relative to image, not anchor
    if subtype=="person":
        target=int(min(W,H)*rng.uniform(0.08,0.18))
    elif subtype=="bird":
        target=int(min(W,H)*rng.uniform(0.025,0.07))
    else:
        target=int(min(W,H)*rng.uniform(0.04,0.12))

    obj=resize_rgba(obj,max(10,target))

    if obj.width>=W or obj.height>=H:
        return None

    # Try positions far from all insulator anchors.
    for _ in range(40):
        px=rng.randint(0,max(0,W-obj.width))
        py=rng.randint(0,max(0,H-obj.height))
        box=[px/W, py/H, (px+obj.width)/W, (py+obj.height)/H]

        bx=(box[0]+box[2])/2; by=(box[1]+box[3])/2
        good=True

        for a in anchors:
            ax=(a[0]+a[2])/2; ay=(a[1]+a[3])/2
            d=math.sqrt((bx-ax)**2+(by-ay)**2)
            if d < 0.33 or box_iou(box,a) > 0:
                good=False
                break

        if not good:
            continue

        out=bg.convert("RGBA")
        out.alpha_composite(obj,(px,py))
        return out.convert("RGB"), box

    return None

def make_contact(ip, foreign_box, subtype, relation, outp):
    im=Image.open(ip).convert("RGB")
    scale=min(1.0,1100/max(im.size))
    if scale<1:
        im=im.resize((int(im.width*scale),int(im.height*scale)))

    W,H=im.size
    head=55
    canvas=Image.new("RGB",(W,H+head),"white")
    canvas.paste(im,(0,head))
    d=ImageDraw.Draw(canvas)

    try:
        font=ImageFont.truetype("DejaVuSans.ttf",16)
    except Exception:
        font=ImageFont.load_default()

    d.text(
        (8,8),
        f"{subtype} | relation={relation} | red=inserted object extent",
        fill="black",
        font=font,
    )

    x1,y1,x2,y2=foreign_box
    rect=[x1*W,y1*H+head,x2*W,y2*H+head]
    d.rectangle(rect,outline="red",width=4)

    Path(outp).parent.mkdir(parents=True,exist_ok=True)
    canvas.save(outp,quality=88)

# =============================================================================
# 1. LOAD SMALL PARENT BACKGROUND POOL
# =============================================================================
bg_tar = get(
    f"{STAGE_ROOT}/parent_background_pool/v6p2e_base_train_background_pool.tar.gz",
    SRC/"background_pool.tar.gz",
)

bg_dir = SRC/"background_pool"
bg_dir.mkdir(parents=True,exist_ok=True)
with tarfile.open(bg_tar,"r:gz") as t:
    t.extractall(bg_dir, filter="data")
bg_tar.unlink(missing_ok=True)

pool_roots=[p for p in bg_dir.rglob("manifest.csv")]
if not pool_roots:
    raise RuntimeError("parent background pool manifest missing")

pool_root=pool_roots[0].parent
df_bg=pd.read_csv(pool_roots[0])

# Resolve relative paths from pool root.
backgrounds=[]
for _,r in df_bg.iterrows():
    ip=pool_root/str(r["image"])
    lp=pool_root/str(r["label"])
    if ip.exists() and lp.exists():
        backgrounds.append({
            "image":ip,
            "label":lp,
            "sha256":str(r["sha256"]),
        })

assert len(backgrounds)>=100, f"Too few backgrounds: {len(backgrounds)}"

# =============================================================================
# 2. SYNTHETIC FOREIGN OBJECTS, ONE OI SUBTYPE AT A TIME
# =============================================================================
(SYN/"images/train").mkdir(parents=True,exist_ok=True)
(SYN/"labels/train").mkdir(parents=True,exist_ok=True)
(QA/"contacts").mkdir(parents=True,exist_ok=True)

synth_rows=[]
rng=random.Random(SEED)

for subtype in OI_SUBTYPES:
    remote=f"{STAGE_ROOT}/open_images/open_images_v7_{subtype}.tar.gz"
    local_tar=SRC/f"oi_{subtype}.tar.gz"

    try:
        get(remote,local_tar)
    except Exception as e:
        print("WARN missing OI subtype",subtype,repr(e))
        continue

    ext=SRC/f"oi_{subtype}_extract"
    if ext.exists():
        shutil.rmtree(ext,ignore_errors=True)
    ext.mkdir(parents=True,exist_ok=True)

    with tarfile.open(local_tar,"r:gz") as t:
        t.extractall(ext, filter="data")
    local_tar.unlink(missing_ok=True)

    # The stage worker packed one directory per subtype.
    dirs=[p for p in ext.iterdir() if p.is_dir()]
    if not dirs:
        shutil.rmtree(ext,ignore_errors=True)
        continue
    oi_root=dirs[0]

    # Open Images class display name
    display_name={
        "balloon":"Balloon",
        "kite":"Kite",
        "plastic_bag":"Plastic bag",
        "bird":"Bird",
        "person":"Person",
    }[subtype]

    importer=fouo.OpenImagesV7DatasetImporter(
        dataset_dir=str(oi_root),
        label_types=["segmentations","detections"],
        classes=[display_name],
        only_matching=True,
        shuffle=False,
    )
    importer.setup()

    object_crops=[]

    try:
        for sample in importer:
            image_path, metadata, labels = sample
            ip=Path(image_path)

            if not ip.exists():
                continue

            if not isinstance(labels,dict):
                continue

            segs=labels.get("segmentations")
            if segs is None or not hasattr(segs,"detections"):
                continue

            im=Image.open(ip).convert("RGB")
            W,H=im.size

            for det in segs.detections:
                if det.label != display_name:
                    continue
                if getattr(det,"mask",None) is None:
                    continue

                x,y,w,h=map(float,det.bounding_box)
                x1=max(0,int(round(x*W)))
                y1=max(0,int(round(y*H)))
                x2=min(W,int(round((x+w)*W)))
                y2=min(H,int(round((y+h)*H)))

                if x2-x1<8 or y2-y1<8:
                    continue

                crop=im.crop((x1,y1,x2,y2)).convert("RGBA")
                mask=np.asarray(det.mask)

                if mask.size == 0:
                    continue

                mask_im=Image.fromarray(
                    (mask>0).astype(np.uint8)*255
                ).resize(crop.size,Image.Resampling.NEAREST)

                crop.putalpha(mask_im)
                object_crops.append(crop)

                if len(object_crops)>=160:
                    break

            if len(object_crops)>=160:
                break
    finally:
        importer.close()

    print(subtype,"masked objects:",len(object_crops))

    if not object_crops:
        shutil.rmtree(ext,ignore_errors=True)
        continue

    # positives
    made_pos=0
    attempts=0

    while made_pos<POS_PER and attempts<POS_PER*30:
        attempts+=1
        bg_rec=rng.choice(backgrounds)
        bg=Image.open(bg_rec["image"]).convert("RGB")
        rows=read_yolo_file(bg_rec["label"])
        anchors=anchor_boxes(rows)

        if not anchors:
            continue

        anchor=rng.choice(anchors)
        obj=rng.choice(object_crops).copy()

        res=place_positive(bg,obj,anchor,subtype,rng)
        if res is None:
            continue

        outim,box=res
        xc,yc,w,h=xyxy_to_yolo(box)

        uid=hashlib.sha256(
            f"pos|{subtype}|{bg_rec['sha256']}|{made_pos}|{SEED}".encode()
        ).hexdigest()[:18]

        di=SYN/"images/train"/f"synpos_{subtype}_{uid}.jpg"
        dl=SYN/"labels/train"/f"synpos_{subtype}_{uid}.txt"

        outim.save(di,quality=92)
        write_yolo(dl,rows+[(4,xc,yc,w,h)])

        synth_rows.append({
            "candidate_type":"synthetic_relation_positive",
            "foreign_subtype":subtype,
            "image":str(di.relative_to(OUT)),
            "label":str(dl.relative_to(OUT)),
            "background_sha256":bg_rec["sha256"],
            "relation":"near_electrical_structure",
            "mapped_class":"foreign_object",
            "train_only":True,
        })

        if made_pos<12:
            make_contact(
                di,box,subtype,"POSITIVE",
                QA/"contacts"/f"pos_{subtype}_{made_pos:03d}.jpg",
            )

        made_pos+=1

    # hard negatives
    made_neg=0
    attempts=0

    while made_neg<NEG_PER and attempts<NEG_PER*40:
        attempts+=1
        bg_rec=rng.choice(backgrounds)
        bg=Image.open(bg_rec["image"]).convert("RGB")
        rows=read_yolo_file(bg_rec["label"])
        anchors=anchor_boxes(rows)

        if not anchors:
            continue

        obj=rng.choice(object_crops).copy()
        res=place_negative(bg,obj,anchors,subtype,rng)

        if res is None:
            continue

        outim,box=res

        uid=hashlib.sha256(
            f"neg|{subtype}|{bg_rec['sha256']}|{made_neg}|{SEED}".encode()
        ).hexdigest()[:18]

        di=SYN/"images/train"/f"synneg_{subtype}_{uid}.jpg"
        dl=SYN/"labels/train"/f"synneg_{subtype}_{uid}.txt"

        outim.save(di,quality=92)

        # Preserve original GT only. Inserted generic object is intentionally unlabeled.
        write_yolo(dl,rows)

        synth_rows.append({
            "candidate_type":"synthetic_relation_negative",
            "foreign_subtype":subtype,
            "image":str(di.relative_to(OUT)),
            "label":str(dl.relative_to(OUT)),
            "background_sha256":bg_rec["sha256"],
            "relation":"far_from_electrical_structure",
            "mapped_class":"background_hard_negative",
            "train_only":True,
        })

        if made_neg<6:
            make_contact(
                di,box,subtype,"HARD_NEGATIVE",
                QA/"contacts"/f"neg_{subtype}_{made_neg:03d}.jpg",
            )

        made_neg+=1

    shutil.rmtree(ext,ignore_errors=True)

df_synth=pd.DataFrame(synth_rows)
df_synth.to_csv(QA/"synthetic_foreign_manifest.csv",index=False)

# =============================================================================
# 3. DIRECT SOURCE MANIFESTS + EXACT-SHA AUDIT
# =============================================================================
parent_manifest_path=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e_split_manifest.csv",
    SRC/"v6p2e_manifest.csv",
)
parent_manifest=pd.read_csv(parent_manifest_path)
parent_sha=set(parent_manifest["sha256"].astype(str))

direct_review=[]
exact_hits=[]

def read_manifest_from_tar(remote, local_name, source_name):
    lp=get(remote,SRC/local_name)

    with tarfile.open(lp,"r:gz") as t:
        members=t.getmembers()
        manifest_member=None

        for m in members:
            if m.isfile() and m.name.endswith("/manifest.csv"):
                manifest_member=m
                break

        if manifest_member is None:
            raise RuntimeError(f"{source_name} manifest.csv not found in tar")

        raw=t.extractfile(manifest_member).read()
        df=pd.read_csv(io.BytesIO(raw))

        # Contact sheets from up to 25 samples by pairing archived images/labels.
        contacts=0

        for _,r in df.head(40).iterrows():
            sha=str(r.get("source_sha256",""))

            if sha and sha in parent_sha:
                exact_hits.append({
                    "source":source_name,
                    "source_sha256":sha,
                    "target_split":str(r.get("target_split","")),
                })

            direct_review.append({
                "candidate_type":"direct_powerline",
                "source":source_name,
                "target_split":str(r.get("target_split","")),
                "source_sha256":sha,
                "mapped_classes":str(r.get("mapped_classes","")),
                "n_boxes":r.get("n_boxes",r.get("boxes","")),
            })

        # Full manifest rows into review table
        if len(df)>40:
            for _,r in df.iloc[40:].iterrows():
                sha=str(r.get("source_sha256",""))

                if sha and sha in parent_sha:
                    exact_hits.append({
                        "source":source_name,
                        "source_sha256":sha,
                        "target_split":str(r.get("target_split","")),
                    })

                direct_review.append({
                    "candidate_type":"direct_powerline",
                    "source":source_name,
                    "target_split":str(r.get("target_split","")),
                    "source_sha256":sha,
                    "mapped_classes":str(r.get("mapped_classes","")),
                    "n_boxes":r.get("n_boxes",r.get("boxes","")),
                })

    lp.unlink(missing_ok=True)
    return df

mpcd_df=read_manifest_from_tar(
    f"{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz",
    "mpcd_stage.tar.gz",
    "MPCD",
)

pe_df=read_manifest_from_tar(
    f"{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz",
    "power_equipment_stage.tar.gz",
    "PowerEquipment",
)

df_exact=pd.DataFrame(exact_hits)
df_exact.to_csv(QA/"direct_vs_v6p2e_exact_sha.csv",index=False)

# =============================================================================
# 4. REVIEW MANIFEST
# =============================================================================
review=[]

for _,r in df_synth.iterrows():
    review.append({
        "candidate_type":r["candidate_type"],
        "source":"OpenImagesV7+V6.2E_train",
        "target_split":"train",
        "foreign_subtype":r["foreign_subtype"],
        "mapped_class":r["mapped_class"],
        "reference":r["image"],
        "recommended_action":"VISUAL_REVIEW_THEN_KEEP",
        "decision":"",
        "reviewer":"",
        "comment":"",
    })

for r in direct_review:
    target=r["target_split"]

    review.append({
        "candidate_type":"direct_powerline",
        "source":r["source"],
        "target_split":target,
        "foreign_subtype":"",
        "mapped_class":r["mapped_classes"],
        "reference":r["source_sha256"],
        "recommended_action":(
            "REVIEW_THEN_KEEP"
            if target=="train"
            else "KEEP_AS_EXTERNAL_EVAL"
        ),
        "decision":"",
        "reviewer":"",
        "comment":"",
    })

df_review=pd.DataFrame(review)
df_review.insert(
    0,
    "review_id",
    [f"V65C{i:06d}" for i in range(len(df_review))]
)
df_review.to_csv(QA/"v6p5_candidate_review_manifest.csv",index=False)

# =============================================================================
# 5. FREEZE CANDIDATE + PACKAGE
# =============================================================================
summary={
    "stage":"v6p5_compose_qa",
    "base":"V6.2E train background pool",
    "synthetic_counts":(
        df_synth.groupby(["candidate_type","foreign_subtype"])
        .size()
        .reset_index(name="count")
        .to_dict("records")
        if len(df_synth) else []
    ),
    "direct_counts":{
        "MPCD":int(len(mpcd_df)),
        "PowerEquipment":int(len(pe_df)),
    },
    "direct_exact_sha_hits_against_V6P2E":int(len(df_exact)),
    "policy":{
        "test_modified":False,
        "OpenImages_source_split":"train",
        "synthetic_positive_label":"foreign_object",
        "synthetic_hard_negative_label":"preserve_background_GT_only",
        "synthetic_background_split":"internal_train_only",
        "auto_promotion":False,
        "near_duplicate_audit_pending_before_final_promotion":True,
    },
}

summary_path=QA/"v6p5_compose_qa_summary.json"
summary_path.write_text(json.dumps(summary,indent=2))

# Content tag from key manifests
tag_input=(
    (QA/"synthetic_foreign_manifest.csv").read_bytes()
    + (QA/"v6p5_candidate_review_manifest.csv").read_bytes()
    + summary_path.read_bytes()
)
tag="v6p5_compose_"+hashlib.sha256(tag_input).hexdigest()[:12]

synthetic_tar=OUT/f"{tag}_synthetic_foreign.tar.gz"
with tarfile.open(synthetic_tar,"w:gz") as t:
    t.add(SYN,arcname="synthetic_foreign")

qa_zip=OUT/f"{tag}_qa.zip"
with zipfile.ZipFile(qa_zip,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in QA.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(QA)))

index_path=OUT/f"{tag}.index.json"
index_path.write_text(json.dumps({
    "tag":tag,
    "synthetic_archive":f"{COMPOSE_ROOT}/{tag}/{synthetic_tar.name}",
    "qa_archive":f"{COMPOSE_ROOT}/{tag}/{qa_zip.name}",
    "source_refs":{
        "MPCD":f"{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz",
        "PowerEquipment":f"{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz",
        "OpenImages":f"{STAGE_ROOT}/open_images/",
        "BackgroundPool":f"{STAGE_ROOT}/parent_background_pool/v6p2e_base_train_background_pool.tar.gz",
    },
    "summary":summary,
},indent=2))

remote_root=f"{COMPOSE_ROOT}/{tag}"

for p in [
    synthetic_tar,
    qa_zip,
    index_path,
    QA/"v6p5_candidate_review_manifest.csv",
    summary_path,
]:
    put(p,f"{remote_root}/{p.name}")

print("TAG:",tag)
print("REMOTE:",remote_root)
print(json.dumps(summary,indent=2))

shutil.rmtree(ROOT,ignore_errors=True)
"""
print("Compose worker prepared:", len(COMPOSE_WORKER), "chars")

In [ ]:
# ==============================================================================
# 2. WRITE WORKER TO RAM + UPLOAD
# ==============================================================================
worker_py = RAM_TMP / "v65_compose_qa_worker.py"

compile(COMPOSE_WORKER, str(worker_py), "exec")
worker_py.write_text(COMPOSE_WORKER)

REMOTE_WORKER = f"{JOB_ROOT}/{worker_py.name}"

r = subprocess.run(
    [CAMBER, "stash", "cp", str(worker_py), REMOTE_WORKER],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)

if r.returncode != 0:
    raise RuntimeError(r.stderr or r.stdout)

worker_py.unlink(missing_ok=True)

print("Worker uploaded:")
print(REMOTE_WORKER)

In [ ]:
# ==============================================================================
# 3. SUBMIT CAMBER JOB
# ==============================================================================
command = (
    "bash -lc 'mkdir -p /tmp/v65compose && "
    f"camber stash cp {REMOTE_WORKER} /tmp/v65compose/worker.py && "
    "python /tmp/v65compose/worker.py'"
)

try:
    import camber

    if not hasattr(camber, "mpi"):
        raise ImportError("camber.mpi unavailable")

    job = camber.mpi.create_job(
        command=command,
        node_size=NODE_SIZE,
        num_nodes=1,
        with_gpu=False,
    )

    print("Submitted Camber compose+QA job:")
    print(job)

except Exception as e:
    print("Automatic submit unavailable:", repr(e))
    print()
    print("Worker đã nằm trên Camber Stash.")
    print("Trong Camber Nova, chạy:")
    print()
    print(
        f"Run {REMOTE_WORKER} as a Python script on one {NODE_SIZE} CPU node. "
        f"It should write outputs under {COMPOSE_ROOT}."
    )

print("\nCommand:")
print(command)

In [ ]:
# ==============================================================================
# 4. CHECK SERVER-SIDE OUTPUTS
# Run this later. It does not download archives.
# ==============================================================================
for target in [
    STAGE_ROOT,
    COMPOSE_ROOT,
]:
    print("\n###", target)

    r = subprocess.run(
        [CAMBER, "stash", "ls", "-r", target],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )

    print((r.stdout or r.stderr)[:20000])

## Khi job xong

Sẽ có một root mới kiểu:

`stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<hash>/`

Trong đó:

- `<tag>_synthetic_foreign.tar.gz`
- `<tag>_qa.zip`
- `<tag>.index.json`
- `v6p5_candidate_review_manifest.csv`
- `v6p5_compose_qa_summary.json`

### Cái cần xem trước khi train

1. `synthetic_foreign_manifest.csv`
2. contact sheets:
   - positive relation
   - hard negative relation
3. `direct_vs_v6p2e_exact_sha.csv`
4. `v6p5_candidate_review_manifest.csv`
5. summary counts theo subtype.

### Sau bước này

Chưa train ngay.

Flow tiếp theo:

`review manifest -> KEEP/REJECT -> final near-duplicate audit -> freeze V6.5 candidate -> YOLO11n sanity train`

Như vậy nếu performance thay đổi, mình biết nó đến từ source nào và không nhập một nồi dữ liệu rồi ngồi đoán.